In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.special import gamma
from numba import jit
from regression_common import ridge_pair, lasso_coefficients, sparse_indices, discrepancy_candidate
from regression_results import BootstrapRun
from regression_spectral import SpectralDesign
from regression_bootstrap import bootstrap_errors, validate_backend
from threadpoolctl import threadpool_limits


In [ ]:
def compute_sigma(Y, X, beta):
    sigma = np.linalg.norm(Y - X @ beta)
    return sigma / np.sqrt(Y.shape[0])


In [ ]:
def compute_norm_cut(k):

    S_truncated = np.zeros_like(S)
    S_truncated[:k] = S[:k]


    Sigma_truncated = np.zeros((U.shape[0], Vt.shape[0]))
    np.fill_diagonal(Sigma_truncated, S_truncated)
    X_truncated = U @ Sigma_truncated @ Vt


    S_cut = Sigma_truncated.T @ Sigma_truncated
    S_inv = np.zeros_like(S_cut)
    S_inv[S_cut != 0] = 1 / S_cut[S_cut != 0]


    beta_cut = Vt.T @ S_inv @ Vt @ X.T @ Y


    norm_cut = np.linalg.norm(beta_cut - beta)

    return norm_cut, k


In [ ]:

def compute_norminf_cut(seed):
    rng = np.random.default_rng(seed)
    e_cut = rng.normal(0, sigma_cut, n)
    Y_cut = np.dot(X, beta_cut) + e_cut
    e_cut_debias = rng.normal(0, sigma_cut_debias, n)
    Y_cut_debias = np.dot(X, beta_cut_debias) + e_cut_debias
    beta_cutstar = Vt.T @ S_inv @ Vt @ X.T @ Y_cut
    beta_cutdestar = Vt.T @ S_inv @ Vt @ X.T @ Y_cut_debias
    # SC debiasing is the identity; retain this branch's bootstrap sample.
    beta_cutstar_debias = beta_cutdestar.copy()
    beta_cutstar[np.abs(beta_cutstar) <= bn_cut] = 0
    beta_cutstar_debias[np.abs(beta_cutstar_debias) <= bn_cut_debias]=0
    norminf_cut = np.linalg.norm(beta_cutstar - beta_cut, ord=np.inf)
    norminf_cut_debias = np.linalg.norm(beta_cutstar_debias - beta_cut_debias, ord=np.inf)
    return norminf_cut, norminf_cut_debias


In [ ]:
beta_names = [
    'beta_cut', 'beta_cut_debias',
]

variable_names = [
    'cut', 'cut_debias',
]


In [ ]:
n        = 1000
p        = 1000
I        = np.eye(p)
C        = 200000
m        = 100
t1       = 2.5e-6
t2       = 5e-6
t3       = 5e-5
t4       = 2.5e-5
t5       = 2.5e-4
max_iter = 10000
k_max    = 1000
kappa    = 0.5
omega    = 3
eta      = 5

i_cut = 0
i_cut_debias = 0
Sigma_cut = 0
Sigma_cut_debias = 0
Inf_norm_cut = 0
Inf_norm_cut_debias = 0
Norm_cut = 0
Norm_cut_debias = 0

# Output controls do not change the estimators or parameter selection.
N = 1000
B = 500
confidence = 0.95
noise_sd = 2.0
seed_root = 20261006
output_root = 'outputs/regression-bootstrap-v2'
resume_from = None

# Execution controls only; statistical settings remain unchanged.
backend = 'cpu'
batch_size = 64
cpu_workers = 4
blas_threads = 8
gpu_devices = (0,)
backend_info = validate_backend(backend, gpu_devices=gpu_devices)


In [ ]:
# Keep the seeded covariance SVD independent of execution threads.
with threadpool_limits(limits=1):
    np.random.seed(24)
    cov_matrix = np.full((p, p), 0.5)
    np.fill_diagonal(cov_matrix, 2.0)

    mean_vector = np.zeros(p)
    X = np.random.multivariate_normal(mean_vector, cov_matrix, n)

    U, S, Vt = np.linalg.svd(X)

    beta = np.zeros(p)


    beta[0:5]   = 2.0   # βi = 2.0, i = 1, 2, 3, 4, 5
    beta[5:10]  = -2.0  # βi = -2.0, i = 6, 7, 8, 9, 10
    beta[10:15] = 1.0   # βi = 1.0, i = 11, 12, 13, 14, 15
    beta[15:20] = -1.0  # βi = -1.0, i = 16, 17, 18, 19, 20

    # Generate 20 random indices
    indices = sparse_indices(np.random, p, size=20)

    # Get the indices of non-zero elements in beta
    nonzero_indices = np.nonzero(beta)[0]
    random_beta = np.zeros(p)
    random_beta[indices] = beta[nonzero_indices]
    beta = random_beta
    spectral_design = SpectralDesign(X, svd=(U, S, Vt))


In [ ]:
run_config = {
    'method': 'SC', 'n': n, 'p': p, 'N': N, 'B': B,
    'sigma': noise_sd, 'confidence': confidence, 'seed_root': seed_root,
    'design_seed': 24, 'noise_model': 'normal',
    'largest_singular_value': float(S[0]),
    'execution': {'backend': backend, 'batch_size': batch_size,
                  'cpu_workers': cpu_workers, 'blas_threads': blas_threads,
                  'gpu_devices': list(gpu_devices)},
    'backend_info': backend_info,
    'preprocess_blas_threads': 1,
    'threshold_grid': [0.001, 0.001, 1.0],
    'threshold_percentile': 50,
    'method_settings': {'selection': 'oracle_rank', 'rank_max': len(S)},
}
with threadpool_limits(limits=blas_threads), BootstrapRun(run_config, X, beta, 'Bootstrapcut.ipynb',
                  output_root=output_root, resume_from=resume_from) as run:
    print('Run directory:', run.directory)
    for x in range(run.start_index, N):
        print(x)
        outer_rng = np.random.default_rng((seed_root, x, 0, 0))



        mean    = 0
        std_dev = noise_sd


        e = outer_rng.normal(mean, std_dev, n)
        norm_e = np.linalg.norm(e)
        Y = np.dot(X, beta) + e

        selection_settings = dict(run_config['method_settings'])
        beta_cut, beta_cut_debias, selected_parameter, selection_diagnostics = spectral_design.select(
            Y, 'SC', selection_settings, beta)
        best_k = int(selected_parameter)
        selected_residual_norm = selection_diagnostics['residual_norm']
        sigma_cut = compute_sigma(Y, X, beta_cut)
        sigma_cut_debias = compute_sigma(Y, X, beta_cut_debias)

        result = []

        for i in range(1, 1001):
            bn = 0.001 * i

            norms = [bn]
            for var_name in beta_names:
                var_copy = globals()[var_name].copy()
                var_copy[np.abs(var_copy) <= bn] = 0
                norm = np.linalg.norm(var_copy - beta)
                norms.append(norm)
            result.append(norms)
        columns = ['bn'] + [f'norm_{name}' for name in beta_names]
        df = pd.DataFrame(result, columns=columns)

        results = []


        for column in df.columns[1:]:
            min_val = df[column].min()
            min_indices = np.where(df[column] == min_val)[0]

            if len(min_indices) > 1 and np.all(np.diff(min_indices) == 1):

                # avg_index = np.mean(min_indices)
                avg_index = np.percentile(min_indices, 50)
                # avg_index = min_indices[0]

            else:

                avg_index = min_indices[0]

            selected_bn = np.interp(avg_index, np.arange(len(df)), df['bn'].to_numpy())
            results.append([min_val, selected_bn])
            results_array = np.array(results)

        for i, name in enumerate(variable_names):
            globals()[f'bn_{name}'] = results_array[i, 1]

        for name in variable_names:
            beta_var = globals()[f'beta_{name}']
            bn_var = globals()[f'bn_{name}']
            beta_var[np.abs(beta_var) <= bn_var] = 0

        for name in variable_names:
            sigma = compute_sigma(Y, X, globals()[f'beta_{name}'])
            globals()[f'sigma_{name}'] = sigma

        Sigma_cut += np.abs(sigma_cut**2 - std_dev**2)
        Sigma_cut_debias += np.abs(sigma_cut_debias**2 - std_dev**2)
        Norm_cut += np.linalg.norm(beta_cut-beta)
        Norm_cut_debias += np.linalg.norm(beta_cut_debias-beta)


        norminf_cut_list = []
        norminf_cut_debias_list = []
        results = bootstrap_errors(
            spectral_design, 'SC', selected_parameter, selection_settings,
            np.column_stack([beta_cut, beta_cut_debias]),
            [sigma_cut, sigma_cut_debias], [bn_cut, bn_cut_debias],
            B, seed_root, x, backend=backend, batch_size=batch_size,
            cpu_workers=cpu_workers, blas_threads=blas_threads, gpu_devices=gpu_devices,
        )
        for norminf_cut, norminf_cut_debias in results:
            norminf_cut_list.append(norminf_cut)
            norminf_cut_debias_list.append(norminf_cut_debias)
        inf_cut = np.linalg.norm(beta - beta_cut, ord=np.inf)
        inf_cut_debias = np.linalg.norm(beta - beta_cut_debias, ord=np.inf)
        Inf_norm_cut += inf_cut
        Inf_norm_cut_debias += inf_cut_debias
        if inf_cut <= np.percentile(norminf_cut_list, 100 * confidence):
            i_cut += 1
        if inf_cut_debias <= np.percentile(norminf_cut_debias_list, 100 * confidence):
            i_cut_debias += 1

        run.record(
            x, selected_parameter=selected_parameter,
            thresholds=[bn_cut, bn_cut_debias],
            estimates=np.column_stack([beta_cut, beta_cut_debias]),
            sigma=[sigma_cut, sigma_cut_debias],
            bootstrap_errors=np.asarray(results),
            threshold_scan=df.to_numpy(), threshold_columns=list(df.columns),
            diagnostics={**selection_diagnostics, 'parameter_kind': 'rank', 'selected_residual_norm': selected_residual_norm},
        )


In [ ]:
# Read all committed replications, including those restored from a checkpoint.
summary = run.summary
i_cut = int(round(summary['coverage_original'] * N))
i_cut_debias = int(round(summary['coverage_debiased'] * N))
Sigma_cut = summary['variance_absolute_error_original'] * N
Sigma_cut_debias = summary['variance_absolute_error_debiased'] * N
Inf_norm_cut = summary['linf_original'] * N
Inf_norm_cut_debias = summary['linf_debiased'] * N
Norm_cut = summary['l2_original'] * N
Norm_cut_debias = summary['l2_debiased'] * N
print(pd.Series(summary, name='SC'))
print('Saved results:', run.directory)
